# 📓 실습 02. 비정형 반도체 CMP 매뉴얼 구조화 파싱 및 청킹

이 실습에서는 정규표현식(Regex)을 이용해 반도체 CMP 공정 정비 지침 매뉴얼 파일(`semiconductor_cmp_manual.md`)을 물리적 헤더 구조에 맞게 단락별로 쪼개고, 상위 계층 정보를 상속받는 정밀 청킹 파이프라인을 구축합니다.

In [ ]:
import os
import re
import json

# 1. 데이터 로드
manual_path = os.path.join("data", "semiconductor_cmp_manual.md")
with open(manual_path, "r", encoding="utf-8") as f:
    content = f.read()

print(f"원본 매뉴얼 파일 크기: {len(content)} 자")

In [ ]:
# 2. 헤더 태그 및 단락 청킹 알고리즘 구현
def parse_markdown_by_headers(text):
    lines = text.split('\n')
    chunks = []
    current_headers = {1: "Root", 2: "", 3: ""}
    current_text = []
    
    header_pattern = re.compile(r'^(#{1,3})\s+(.*)$')
    
    for line in lines:
        match = header_pattern.match(line)
        if match:
            # 이전까지 쌓였던 텍스트 블록을 청크로 저장
            if current_text:
                chunks.append({
                    "headers": [current_headers[1], current_headers[2], current_headers[3]],
                    "content": "\n".join(current_text).strip()
                })
                current_text = []
            
            level = len(match.group(1))
            title = match.group(2).strip()
            current_headers[level] = title
            # 하위 레벨 초기화
            for l in range(level + 1, 4):
                current_headers[l] = ""
        else:
            current_text.append(line)
            
    # 마지막 잔류 텍스트 저장
    if current_text:
        chunks.append({
            "headers": [current_headers[1], current_headers[2], current_headers[3]],
            "content": "\n".join(current_text).strip()
        })
        
    # 내용이 비어있는 무의미한 청크는 필터링
    return [c for c in chunks if len(c["content"]) > 10]

parsed_chunks = parse_markdown_by_headers(content)
print(f"파싱 완료! 총 {len(parsed_chunks)} 개의 구조적 청크가 생성되었습니다.\n")

# 첫 번째 청크 예시 출력
print("=== [청크 샘플 1] ===")
print(f"상위 계층: {parsed_chunks[0]['headers']}")
print(f"본문 내용:\n{parsed_chunks[0]['content'][:150]}...")

In [ ]:
# 3. 파싱 결과 저장
os.makedirs("data", exist_ok=True)
output_path = os.path.join("data", "parsed_chunks.json")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(parsed_chunks, f, indent=2, ensure_ascii=False)
print(f"성공적으로 파싱 데이터를 파일로 저장했습니다: {output_path}")